# MotionForge — Treinamento multimodal

Notebook para executar pelo navegador do celular usando Google Colab. Selecione **Runtime → Change runtime type → GPU** quando houver GPU disponível. O Colab pode limitar ou desconectar sessões gratuitas.

O treino ajusta apenas os adaptadores de busca visual sobre um encoder OpenCLIP congelado. Não cria um gerador de imagens/vídeos/3D do zero. O arquivo de pares e todos os caminhos de mídia precisam estar disponíveis no Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DATA_DIR = Path('/content/drive/MyDrive/MotionForgeData')
PAIRS_FILE = DATA_DIR / 'pairs.jsonl'
CHECKPOINT_FILE = DATA_DIR / 'visual_adapter.pt'
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Pairs:', PAIRS_FILE)
print('Checkpoint:', CHECKPOINT_FILE)
assert PAIRS_FILE.is_file(), 'Coloque pairs.jsonl e os arquivos de imagem correspondentes em MotionForgeData.'

In [ ]:
!git clone --depth 1 --branch motionforge-multimodal-learning https://github.com/japaguimarassi-create/J.git /content/J
%cd /content/J
!python -m pip install -r multimodal/requirements.txt

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Sem GPU nesta sessão. O treino pode ficar muito lento em CPU.')

In [ ]:
!python multimodal/train_visual.py --pairs "{PAIRS_FILE}" --epochs 20 --batch-size 16 --workers 2 --max-minutes 180 --output "{CHECKPOINT_FILE}" --resume

## Continuar o treinamento

Abra uma nova sessão, monte o mesmo Drive e execute novamente a célula de treinamento. O argumento `--resume` carrega os adaptadores e o estado do otimizador salvos no checkpoint. Para uma nova rodada mais longa, aumente `--epochs` e/ou `--max-minutes`. Se ocorrer falta de memória, reduza `--batch-size` para 8 ou 4.

In [ ]:
!python multimodal/search_visual.py --pairs "{PAIRS_FILE}" --checkpoint "{CHECKPOINT_FILE}" --query "a cinematic 3D character with dramatic rim lighting" --top-k 12